# Tuvan (tyv) — Tokenisation and Translation

Tuvan (Siberian branch, Cyrillic script) currently has rule-based tokenisation and prototype-quality morphological analysis. NLLB-200 provides cross-lingual embeddings and machine translation.

In [ ]:
# Install TurkicNLP
# pip install turkicnlp          # core (tokenization, transliteration)
# pip install "turkicnlp[stanza]"  # adds POS, lemma, depparse, NER
# pip install "turkicnlp[nllb]"    # adds cross-lingual embeddings + translation
# pip install "turkicnlp[all]"     # all optional dependencies

In [ ]:
import turkicnlp
from turkicnlp import Pipeline

## 1. Tokenisation

In [ ]:
from turkicnlp.scripts import Script
from turkicnlp.scripts.detector import detect_script
from turkicnlp.scripts.transliterator import Transliterator

# Tuvan Cyrillic text
cyrl = "Мен школага баар мен."
print("Script Detection:")
print(f"  Detected: {detect_script(cyrl).name}")
print()

# Cyrillic -> Turkic Common Alphabet (Latin)
try:
    t = Transliterator("tyv", source=Script.CYRILLIC, target=Script.COMMON_TURKIC)
    common = t.transliterate(cyrl)
    print(f"Cyrillic:        {cyrl}")
    print(f"Turkic Common:   {common}")
    
    # Reverse: Turkic Common -> Cyrillic
    t_back = Transliterator("tyv", source=Script.COMMON_TURKIC, target=Script.CYRILLIC)
    cyrl_restored = t_back.transliterate(common)
    print(f"Restored:        {cyrl_restored}")
    print(f"Round-trip match: {cyrl == cyrl_restored}")
except Exception as e:
    print(f"⚠ Note: Transliteration to Turkic Common Alphabet (Latin) may not be fully supported for Tuvan: {e}")
    print("  For Cyrillic-based languages, use Script.LATIN as alternative")

## 2. Script Detection and Cyrillic ↔ Latin Transliteration

Tuvan uses Cyrillic script. The transliteration system enables conversion to Latin for processing.

In [ ]:
turkicnlp.download("tyv")
nlp_tok = Pipeline("tyv", processors=["tokenize"])
doc = nlp_tok("Мен школага баар мен.")
print([w.text for w in doc.words])

## 2. Morphological Analysis (Apertium FST — Prototype)

In [ ]:
nlp = Pipeline(
    "tyv",
    processors=["tokenize", "morph"],
    morph_backend="apertium",
)
doc = nlp("Мен школага баар мен.")
for w in doc.words:
    print(f"{w.text:<18} lemma={w.lemma} feats={w.feats}")

## 3. Translation via NLLB-200

In [ ]:
turkicnlp.download("tyv", processors=["translate"])
trans = Pipeline("tyv", processors=["translate"], translate_tgt_lang="eng_Latn")
doc = trans("Мен школага баар мен.")
print("EN:", doc.translation)